# Data Refresh

**DB location:** `listen-wiseer/data/listen_wiseer.db`

You can also open it directly with any DuckDB client:
```bash
duckdb data/listen_wiseer.db
```

Or query it from Python anywhere:
```python
import duckdb
conn = duckdb.connect('data/listen_wiseer.db', read_only=True)
conn.execute('SELECT * FROM track_profile LIMIT 5').df()
```

**Tables:** `tracks`, `audio_features`, `artists`, `genre_map`, `playlist_tracks`, `track_artists`, `faves`  
**View:** `track_profile` — enriched join of all tables (use this for analysis/ML)

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from utils.logging import configure_logging

configure_logging()

from etl.db import get_connection, init_schema

conn = get_connection(read_only=True)
print(f"DB: {ROOT / 'data/listen_wiseer.db'}")

In [ ]:
import os
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "etl.sync"],
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
    cwd=str(ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print("ERRORS:", result.stderr)

## Current DB state

In [ ]:
for table in ["tracks", "audio_features", "artists", "genre_map", "playlist_tracks", "faves"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:20s}: {n:,}")

## Track profile sample

In [ ]:
conn.execute("""
    SELECT track_name, gen_8, energy, danceability, valence, fave_score
    FROM track_profile
    WHERE gen_8 IS NOT NULL
    ORDER BY fave_score DESC, energy DESC
    LIMIT 10
""").df()

## Genre breakdown

In [ ]:
conn.execute("""
    SELECT gen_8, COUNT(*) as tracks,
           ROUND(AVG(energy), 2) as avg_energy,
           ROUND(AVG(danceability), 2) as avg_dance
    FROM track_profile
    WHERE gen_8 IS NOT NULL
    GROUP BY gen_8
    ORDER BY tracks DESC
""").df()

## Playlists

In [ ]:
conn.execute("""
    SELECT p.playlist_name, COUNT(*) as n_tracks, p.gen_8
    FROM playlist_tracks pt
    JOIN playlists p USING (playlist_id)
    GROUP BY p.playlist_name, p.gen_8
    ORDER BY n_tracks DESC
""").df()

---
## Sync fresh data from Spotify API

Run cells below one at a time. Each step prints progress and skips data already in the DB.
Requires `.spotify_cache` (run `make mcp-server` once to authenticate first).

Steps:
1. Fetch playlists — configure which ones to include/exclude
2. Fetch tracks per playlist — skips playlists with no new tracks
3. Upsert tracks + playlist_tracks
4. Fetch audio features — only for tracks not already in DB
5. Fetch artist features — only for artists not already in DB
6. Verify final counts

In [ ]:
# ── Sync setup ────────────────────────────────────────────────────────────────
# Close the read-only connection opened above, open a writable one
conn.close()

from etl.db import get_connection
from spotify.client import SpotifyClient
from spotify.fetch import fetch_my_playlists

write_conn = get_connection(read_only=False)
init_schema(write_conn)
client = SpotifyClient()
print("Connected (read-write) and Spotify client ready.")

In [ ]:
# ── Step 1: Fetch & register playlists ───────────────────────────────────────
# Fetches all playlists from Spotify, upserts into the DB (syncing names for
# renames), and seeds include_in_refresh=TRUE for known playlists from const.
# Then filters to only playlists marked for refresh.

from utils.const import playlists as KNOWN_PLAYLISTS  # {id: name} from old registry

raw_playlists = fetch_my_playlists(client)

# Upsert all — INSERT new rows, UPDATE name on existing (handles renames)
for p in raw_playlists:
    write_conn.execute(
        """
        INSERT INTO playlists (playlist_id, playlist_name, include_in_refresh)
        VALUES (?, ?, TRUE)
        ON CONFLICT (playlist_id) DO UPDATE SET playlist_name = excluded.playlist_name
    """,
        [p["id"], p["name"]],
    )

# Ensure all known const IDs are flagged for refresh (idempotent seed)
for pid in KNOWN_PLAYLISTS:
    write_conn.execute(
        """
        INSERT INTO playlists (playlist_id, playlist_name, include_in_refresh)
        VALUES (?, ?, TRUE)
        ON CONFLICT (playlist_id) DO UPDATE SET include_in_refresh = TRUE
    """,
        [pid, KNOWN_PLAYLISTS[pid]],
    )

# Show full registry
all_registered = write_conn.execute("""
    SELECT playlist_id, playlist_name, include_in_refresh
    FROM playlists ORDER BY include_in_refresh DESC, playlist_name
""").df()
print(f"Registered playlists: {len(all_registered)}")
print(all_registered.to_string(index=False))

# Filter to active ones
active_ids = set(
    write_conn.execute("SELECT playlist_id FROM playlists WHERE include_in_refresh = TRUE").df()[
        "playlist_id"
    ]
)
selected = [p for p in raw_playlists if p["id"] in active_ids]
print(f"\nSelected for sync: {len(selected)} / {len(raw_playlists)}")
print("\nTo exclude a playlist from future syncs:")
print(
    "  write_conn.execute(\"UPDATE playlists SET include_in_refresh=FALSE WHERE playlist_name='...'\")"
)

In [ ]:
# ── Step 2: Fetch tracks per playlist ────────────────────────────────────────
# Upserts playlists + playlist_tracks. Collects all track objects for steps 3-5.
import polars as pl

from spotify.fetch import fetch_playlist_tracks

# Load track IDs already in DB to detect what's new
existing_track_ids = set(
    row[0] for row in write_conn.execute("SELECT track_id FROM tracks").fetchall()
)
print(f"Tracks already in DB: {len(existing_track_ids):,}")

all_tracks = []  # TrackFeatures for ALL selected tracks
new_track_ids = set()  # IDs not yet in DB

for p in selected:
    pid, pname = p["id"], p["name"]

    write_conn.execute(
        "INSERT OR IGNORE INTO playlists (playlist_id, playlist_name) VALUES (?, ?)",
        [pid, pname],
    )

    tracks = fetch_playlist_tracks(client, pid)
    all_tracks.extend(tracks)

    if tracks:
        pt = pl.DataFrame([{"playlist_id": pid, "track_id": t.id} for t in tracks])
        write_conn.execute("INSERT OR IGNORE INTO playlist_tracks SELECT * FROM pt")

    new_here = [t for t in tracks if t.id not in existing_track_ids]
    new_track_ids.update(t.id for t in new_here)
    print(f"  {pname:<35} {len(tracks):>4} tracks  ({len(new_here)} new)")

print(f"\nTotal tracks across selected playlists : {len(all_tracks):,}")
print(f"New tracks (not yet in DB)             : {len(new_track_ids):,}")

In [ ]:
# ── Step 3: Upsert track + track_artists rows ─────────────────────────────────
# Only inserts tracks not already in DB.
_KEY_MAP = {
    0: "C",
    1: "Db",
    2: "D",
    3: "Eb",
    4: "E",
    5: "F",
    6: "F#",
    7: "G",
    8: "Ab",
    9: "A",
    10: "Bb",
    11: "B",
}
_MODE_MAP = {0: "Minor", 1: "Major"}

new_tracks = [t for t in all_tracks if t.id in new_track_ids]

if not new_tracks:
    print("No new tracks — nothing to upsert.")
else:
    track_rows = pl.DataFrame(
        [
            {
                "track_id": t.id,
                "track_name": t.name,
                "release_date": t.release_date,
                "year": int(t.release_date[:4])
                if t.release_date and len(t.release_date) >= 4
                else None,
                "decade": None,
                "popularity": None,
                "first_genre": None,
                "genre_cat": None,
            }
            for t in new_tracks
        ]
    ).unique("track_id")
    write_conn.execute("INSERT OR IGNORE INTO tracks SELECT * FROM track_rows")
    print(f"Upserted {len(track_rows):,} new track rows.")

    ta_rows = [{"track_id": t.id, "artist_id": aid} for t in new_tracks for aid in t.artist_ids]
    if ta_rows:
        ta = pl.DataFrame(ta_rows).unique(["track_id", "artist_id"])
        write_conn.execute("INSERT OR IGNORE INTO track_artists SELECT * FROM ta")
        print(f"Upserted {len(ta):,} track_artist rows.")

In [ ]:
# ── Step 4: Audio features ────────────────────────────────────────────────────
# Only fetches for tracks not already in audio_features.
from spotify.fetch import fetch_audio_features

existing_audio_ids = set(
    row[0] for row in write_conn.execute("SELECT track_id FROM audio_features").fetchall()
)
all_track_ids = list({t.id for t in all_tracks})
missing_audio_ids = [tid for tid in all_track_ids if tid not in existing_audio_ids]

print(f"Tracks with audio features in DB : {len(existing_audio_ids):,}")
print(f"Tracks missing audio features    : {len(missing_audio_ids):,}")

if not missing_audio_ids:
    print("Nothing to fetch.")
else:
    audio = fetch_audio_features(client, missing_audio_ids)
    if audio:
        af_rows = pl.DataFrame(
            [
                {
                    "track_id": a.id,
                    "danceability": a.danceability,
                    "energy": a.energy,
                    "loudness": a.loudness,
                    "speechiness": a.speechiness,
                    "acousticness": a.acousticness,
                    "instrumentalness": a.instrumentalness,
                    "liveness": a.liveness,
                    "valence": a.valence,
                    "tempo": a.tempo,
                    "duration_ms": a.duration_ms,
                    "time_signature": a.time_signature,
                    "key": a.key,
                    "mode": a.mode,
                    "key_labels": _KEY_MAP.get(a.key, ""),
                    "mode_labels": _MODE_MAP.get(a.mode, ""),
                    "key_mode": f"{_KEY_MAP.get(a.key, '')} {_MODE_MAP.get(a.mode, '')}",
                }
                for a in audio
            ]
        ).unique("track_id")
        write_conn.execute("INSERT OR IGNORE INTO audio_features SELECT * FROM af_rows")
        print(f"Inserted {len(af_rows):,} audio feature rows.")

In [ ]:
# ── Step 5: Artist features ───────────────────────────────────────────────────
# Only fetches for artists not already in artists table.
from spotify.fetch import fetch_artist_features

existing_artist_ids = set(
    row[0] for row in write_conn.execute("SELECT artist_id FROM artists").fetchall()
)
all_artist_ids = list({aid for t in all_tracks for aid in t.artist_ids})
missing_artist_ids = [aid for aid in all_artist_ids if aid not in existing_artist_ids]

print(f"Artists with data in DB  : {len(existing_artist_ids):,}")
print(f"Artists missing data     : {len(missing_artist_ids):,}")

if not missing_artist_ids:
    print("Nothing to fetch.")
else:
    artists = fetch_artist_features(client, missing_artist_ids)
    if artists:
        ar_rows = pl.DataFrame(
            [
                {"artist_id": a.id, "popularity": a.popularity, "genres": str(a.genres)}
                for a in artists
            ]
        ).unique("artist_id")
        write_conn.execute("INSERT OR IGNORE INTO artists SELECT * FROM ar_rows")
        print(f"Inserted {len(ar_rows):,} artist rows.")

In [ ]:
# ── Step 6: Verify ────────────────────────────────────────────────────────────
write_conn.close()

# Re-open read-only for inspection
conn = get_connection(read_only=True)

for table in [
    "playlists",
    "tracks",
    "audio_features",
    "artists",
    "track_artists",
    "playlist_tracks",
]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<20}: {n:,}")

n_profile = conn.execute("SELECT COUNT(*) FROM track_profile").fetchone()[0]
print(f"\ntrack_profile view       : {n_profile:,} rows")
print("Sync complete.")